# PySpark Test Suite — CSV Edition
Make sure `employees.csv` is in the same folder as this notebook.

## Setup

In [1]:
import os
import sys
import tempfile

# Fix Windows Python version mismatch
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

print(f"Python: {sys.executable}")

Python: c:\Users\alixcover\anaconda3\envs\pyspark-env\python.exe


In [2]:
# Spark session
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("pyspark-csv-test")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark

In [3]:
# Path to CSV — adjust if your file is elsewhere
CSV_PATH = "ratings.csv"

assert os.path.exists(CSV_PATH), f"File not found: {CSV_PATH}"
print(f"CSV found: {os.path.abspath(CSV_PATH)}")

CSV found: c:\Users\alixcover\Desktop\documents personel\laplateforme source de projet\sparkle-movie\ratings.csv


## 1. Read CSV & Inspect

In [4]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
)

print(f"Rows: {df.count()}  |  Columns: {df.columns}")
df.printSchema()
df.show()

Rows: 32000204  |  Columns: ['userId', 'movieId', 'rating', 'timestamp']
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|     17|   4.0|944249077|
|     1|     25|   1.0|944250228|
|     1|     29|   2.0|943230976|
|     1|     30|   5.0|944249077|
|     1|     32|   5.0|943228858|
|     1|     34|   2.0|943228491|
|     1|     36|   1.0|944249008|
|     1|     80|   5.0|944248943|
|     1|    110|   3.0|943231119|
|     1|    111|   5.0|944249008|
|     1|    161|   1.0|943231162|
|     1|    166|   5.0|943228442|
|     1|    176|   4.0|944079496|
|     1|    223|   3.0|944082810|
|     1|    232|   5.0|943228442|
|     1|    260|   5.0|943228696|
|     1|    302|   4.0|944253272|
|     1|    306|   5.0|944248888|
|     1|    307|   5.0|944253207|
|     1|    32

## 2. Aggregations

In [ ]:
agg_df = (
    df.groupBy("userId")
    .agg(
        F.count("movieId").alias("num_ratings"),
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.max("rating").alias("max_rating"),
        F.min("rating").alias("min_rating"),
    )
    .orderBy("num_ratings", ascending=False)
)

agg_df.show()

{"ts": "2026-03-17 13:56:41.013", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `department` cannot be resolved. Did you mean one of the following? [`rating`, `userId`, `movieId`, `timestamp`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o47.agg.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `department` cannot be resolved. Did you mean one of the following? [`rating`, `userId`, `movieId`, `timestamp`]. SQLSTATE: 42703;\n'Aggregate ['department], ['department, 'count('name) AS headcount#47, 'round('avg('salary), 2) AS avg_salary#48, 'max('salary) AS max_salary#

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `department` cannot be resolved. Did you mean one of the following? [`rating`, `userId`, `movieId`, `timestamp`]. SQLSTATE: 42703;
'Aggregate ['department], ['department, 'count('name) AS headcount#47, 'round('avg('salary), 2) AS avg_salary#48, 'max('salary) AS max_salary#49, 'min('salary) AS min_salary#50]
+- Relation [userId#17,movieId#18,rating#19,timestamp#20] csv


## 3. Spark SQL

In [ ]:
df.createOrReplaceTempView("employees")

sql_result = spark.sql("""
    SELECT department,
           COUNT(*)              AS headcount,
           SUM(salary)           AS total_salary,
           ROUND(AVG(salary), 2) AS avg_salary
    FROM employees
    GROUP BY department
    ORDER BY total_salary DESC
""")

sql_result.show()

## 4. UDF — Salary Bands

In [ ]:
@F.udf(StringType())
def salary_band(salary: int) -> str:
    if salary >= 90000:
        return "Senior"
    elif salary >= 80000:
        return "Mid"
    return "Junior"

df.select("name", "department", "salary", salary_band("salary").alias("band")).show()

## 5. Pandas Interop

In [ ]:
# Spark -> Pandas
pdf = df.toPandas()
print(f"Pandas DataFrame shape: {pdf.shape}")
display(pdf.describe())

# Pandas -> Spark round-trip
sdf = spark.createDataFrame(pdf)
print(f"\nRound-trip row count: {sdf.count()}")

## 6. Filtering + Computed Columns

In [ ]:
high_earners = (
    df.filter(F.col("salary") >= 85000)
    .withColumn("bonus", F.round(F.col("salary") * 0.10, 2))
    .orderBy(F.col("salary").desc())
)

print(f"High earners (>= 85k): {high_earners.count()} employees")
high_earners.show()

## 7. Parquet Round-Trip

In [ ]:
parquet_path = os.path.join(tempfile.mkdtemp(), "employees.parquet")

df.write.mode("overwrite").parquet(parquet_path)
loaded = spark.read.parquet(parquet_path)

print(f"Written to: {parquet_path}")
print(f"Original rows: {df.count()}  |  Loaded rows: {loaded.count()}")
loaded.show(5)

## Cleanup

In [ ]:
spark.stop()
print("Spark session stopped.")